# From Hackathon AI to Production AI

## Chapter 01 — Do We Understand It?

Version 2.0

## Story So Far


It is now Monday morning.

The HelpDeskAI prototype won the hackathon.

Management has approved a pilot deployment.

The engineering team celebrates briefly before a senior
engineer asks a simple question.

"Can anyone explain why the model predicted 'Billing'
for this ticket?"

The room becomes quiet.

The application works.

Nobody understands how it behaves.

## Workshop Context


In Chapter 00 we proved that Gemini could solve our
classification problem.

Today we ask a different engineering question.

Can we observe, understand and reproduce the behaviour
of our AI system?

Without observability there is no debugging.

Without debugging there is no engineering.

## Learning Outcomes

- Log prompts sent to Gemini
- Capture model responses
- Measure response latency
- Build an experiment log
- Understand why observability is essential
- Think beyond successful demos

## Today's Engineering Question



Can we understand how our AI system behaves?

If the model changes its answer tomorrow,

would we know why?

## Engineering Principle



You cannot improve

what you cannot observe.

## Looking Back


Yesterday's prototype answered customer tickets.

Today's engineer asks different questions.

• Which prompt produced this answer?

• Which model version was used?

• How long did it take?

• Was the answer consistent?

• Can we reproduce yesterday's experiment?

Engineering begins with curiosity.

Observability makes curiosity actionable.

## Engineering Discussion



Imagine the following conversation.

Manager:

"The demo looked great."

Engineer:

"Thank you."

Manager:

"Can you explain why the third ticket was classified
as Technical?"

Engineer:

"I don't know."

Manager:

"Can you reproduce yesterday's result?"

Engineer:

"I'm not sure."

This conversation happens surprisingly often.

Many AI applications answer questions.

Very few explain themselves.

## Today's Lab


Throughout today's notebook we will enhance our prototype.

We will not improve the model.

Instead,

we will improve our visibility into the system.

Our goal is to transform a black box into an observable
application.

## Hands-on Exercise 1

### Import Required Libraries


We begin by importing the libraries required for today's
experiments.

Notice that we now include modules for measuring time
and recording structured data.

In [ ]:
import json
import time
from datetime import datetime

from google import genai

## Hands-on Exercise 2

### Create Gemini Client


Reuse the API key from the previous chapter.

Create the Gemini client that will be used throughout
today's experiments.

In [ ]:
try:
    from google.colab import userdata

    API_KEY = userdata.get("GOOGLE_API_KEY")

except Exception:

    API_KEY = input(
        "Enter your Gemini API Key: "
    ).strip()

client = genai.Client(
    api_key=API_KEY
)

print("Client ready.")

## Reflection


Notice that our application is still functioning exactly as
it did in Chapter 00.

The difference is not the model.

The difference is the engineer.

Instead of simply accepting an answer, we now want to know

- what was asked,
- when it was asked,
- how long it took,
- and what was returned.

Observability begins with recording evidence.

## Hands-on Exercise 3

### Build an Experiment Logger


Rather than printing values to the screen, create a reusable
function that records every interaction with the model.

Every experiment should answer the questions:

• What prompt was used?

• When was it executed?

• How long did it take?

• What response was returned?

In [ ]:
experiment_log = []

def classify_ticket(ticket):

    prompt = f'''
You are a customer support classifier.

Return only one category.

Billing
Technical
Account

Ticket

{ticket}
'''

    start = time.perf_counter()

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
    )

    elapsed = time.perf_counter() - start

    record = {

        "timestamp": datetime.now().isoformat(),

        "ticket": ticket,

        "prediction": response.text.strip(),

        "latency_seconds": round(elapsed, 3),

        "model": "gemini-2.5-flash"

    }

    experiment_log.append(record)

    return record

## Engineering Discussion


Notice that we have not changed the prompt.

We have not changed Gemini.

We have not changed the prediction.

We have changed the visibility of our application.

Professional software engineering often involves improving
the system around the model rather than the model itself.

## Hands-on Exercise 4

### Run Your First Logged Experiment


Execute the classifier and inspect the returned record.

In [ ]:
result = classify_ticket(
    "My password reset link has expired."
)

result

## Reflection


Instead of receiving only a prediction, we now have
structured information describing the interaction.

This structure makes future analysis possible.

## Hands-on Exercise 5

### Run Multiple Experiments


Observe whether latency and predictions remain consistent.

In [ ]:
tickets = [

    "My invoice contains the wrong amount.",

    "The mobile application crashes on startup.",

    "Please update my registered email address.",

    "I cannot log into my account.",

    "My payment failed yesterday."

]

for ticket in tickets:

    classify_ticket(ticket)

len(experiment_log)

## Hands-on Exercise 6

### Inspect the Experiment Log


Every interaction is now stored as structured data.

Inspect the first few records.

In [ ]:
for record in experiment_log:

    print(record)

    print("-" * 60)

## Think Like an Engineer


A successful AI application is not simply one that returns
correct answers.

It is one whose behaviour can be understood.

Logging converts observations into evidence.

Evidence enables debugging.

Debugging enables improvement.

## Hands-on Exercise 7

### Export the Experiment Log


Persist experiment results so they can be analysed later.

This simple step is the foundation of evaluation pipelines.

In [ ]:
with open(
    "experiment_log.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        experiment_log,
        f,
        indent=2
    )

print("experiment_log.json written.")

## Engineering Discussion


Suppose a customer reports that the AI made an incorrect
classification yesterday.

Without logs, the engineering team has no evidence.

With logs, the team can reconstruct the interaction,
inspect the prompt, review the prediction, and begin
investigating the issue.

Observability transforms anecdotes into data.

## Challenge Exercise


Extend the experiment log by recording additional
information such as:

- prompt length
- ticket length
- response length
- execution date
- model version

Which of these metrics might become useful as your
application grows?

## Hands-on Exercise 8

### Analyze Your Experiments


Good engineers don't just collect data—they analyze it.

Let's compute a few simple statistics from our experiment log.

In [ ]:
latencies = [
    record["latency_seconds"]
    for record in experiment_log
]

print(f"Experiments      : {len(latencies)}")
print(f"Minimum Latency  : {min(latencies):.3f} seconds")
print(f"Maximum Latency  : {max(latencies):.3f} seconds")
print(f"Average Latency  : {sum(latencies)/len(latencies):.3f} seconds")

## Hands-on Exercise 9

### Count Model Predictions


Look for patterns in the model outputs.

Simple summaries often reveal surprising behaviour.

In [ ]:
from collections import Counter

counts = Counter(
    record["prediction"]
    for record in experiment_log
)

print("Prediction Summary")

for label, count in counts.items():
    print(f"{label:12} {count}")

## Reflection


Today we measured only a handful of experiments.

Production systems may process thousands of requests every
minute.

Observability allows engineers to identify trends instead of
isolated incidents.

The goal is not simply to answer one question.

The goal is to understand system behaviour over time.

## Engineering Discussion


Logging alone is not enough.

Consider these questions.

• Did the model become slower today?

• Is one ticket category frequently misclassified?

• Did a prompt modification improve quality?

• Did a new model version change behaviour?

These questions cannot be answered unless observations are
collected consistently.

## Hands-on Exercise 10

### Compare Prompt Variations


Experiment with small prompt changes.

Observe whether the prediction changes.

Record your observations in the experiment log.

In [ ]:
prompt_versions = [

    "Return only one category.",

    "Return exactly one category and nothing else.",

    "Classify this customer support ticket."

]

ticket = "My monthly payment was charged twice."

for version in prompt_versions:

    prompt = f'''
{version}

Categories

Billing
Technical
Account

Ticket

{ticket}
'''

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
    )

    print(version)
    print(response.text)
    print("-" * 60)

## Think Like an Engineer


Changing prompts without recording results is experimentation.

Recording observations, comparing outcomes, and making
evidence-based decisions is engineering.

Engineering replaces intuition with measurement.

## Checkpoint


At this point your HelpDeskAI prototype can

• classify support tickets

• record every interaction

• measure response latency

• save experiment logs

• compare prompt variations

Your application has become observable.

## Chapter Summary


Chapter 01 shifted our attention away from prompts and toward
engineering practices.

We learned that successful AI systems must be observable.

By logging prompts, responses, execution time, and experiment
metadata, we created the foundation needed to evaluate,
debug, and improve our application.

Observability is one of the first characteristics of a
production-ready AI system.

## Production Readiness Checklist

Completed

☑ Prompt logging
☑ Response logging
☑ Latency measurement
☑ Experiment records
☑ Structured JSON logs
☑ Basic experiment analysis
☑ Prompt comparison

Still Missing

☐ Error handling
☐ Retries
☐ Rate-limit handling
☐ Timeout recovery
☐ Evaluation dataset
☐ Automated testing
☐ Monitoring dashboard
☐ Deployment

## Looking Ahead

### Chapter 02 — Will It Survive Reality?


Everything has worked perfectly so far.

Real production systems are not so cooperative.

Networks fail.

APIs become unavailable.

Users submit unexpected input.

Rate limits are exceeded.

In the next chapter we stop assuming success and begin
designing software that survives failure.

## Reflection


Chapter 00 asked

Can it answer?

Chapter 01 asked

Do we understand it?

Answering questions is intelligence.

Understanding behaviour is engineering.

Production AI requires both.